# Message State

> **Source:** `repo1/langgraph_core.py`

Demo Message State


## Imports and Setup


In [ ]:
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict, Annotated
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
import operator
from dotenv import load_dotenv
load_dotenv()
class SimpleState(TypedDict):
    input: str
    output: str
    step: int
class AccumulatingState(TypedDict):
    messages: Annotated[list[str], operator.add]  # lists concatenate when merged
    count: Annotated[int, operator.add]  # counts sum when merged
from langgraph.graph import add_messages
class MessageState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
class MultiStepState(TypedDict):
    input: str
    analyzed: str
    enhanced: str
    final: str


## Implementation


In [ ]:
def demo_message_state():
    llm = init_chat_model("gpt-4o-mini", temperature=0)

    def chat_node(state: MessageState) -> dict:
        response = llm.invoke(state["messages"])
        return {"messages": [response]}

    graph = StateGraph(MessageState)
    graph.add_node("chat_node", chat_node)
    graph.add_edge(START, "chat_node")
    graph.add_edge("chat_node", END)

    app = graph.compile()

    result = app.invoke({"messages": [HumanMessage(content="Say Hello in Tagalog")]})

    print("\nMessage State Result:")
    for msg in result["messages"]:
        role = "Human" if isinstance(msg, HumanMessage) else "AI"
        print(f"  {role}: {msg.content}")


## Execute Demo


In [ ]:
demo_message_state()
